**CI twin of `ch19-pca.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

feats = ["bill_length_mm", "bill_depth_mm",
         "flipper_length_mm", "body_mass_g"]
df = load_csv("penguins").dropna(subset=feats)

X2 = StandardScaler().fit_transform(df[["flipper_length_mm",
                                        "body_mass_g"]])
pca2 = PCA().fit(X2)
print("variance carried:", np.round(pca2.explained_variance_ratio_, 3))

fig, ax = plt.subplots(figsize=(4.6, 4))
ax.scatter(X2[:, 0], X2[:, 1], s=8, alpha=0.4)
for comp, ratio, name in zip(pca2.components_,
                             pca2.explained_variance_ratio_,
                             ("PC1", "PC2")):
    ax.annotate("", xy=comp * ratio * 3.5, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", lw=2, color="red"))
    ax.annotate(name, xy=comp * ratio * 3.8, fontsize=9)
ax.set_xlabel("flipper (scaled)")
ax.set_ylabel("mass (scaled)")
ax.set_aspect("equal")
plt.show()

In [ ]:
scaler = StandardScaler().fit(df[feats])
Xs = scaler.transform(df[feats])

pca = PCA().fit(Xs)
ratios = pca.explained_variance_ratio_
print("per-axis:   ", np.round(ratios, 3))
print("cumulative: ", np.round(np.cumsum(ratios), 3))

In [ ]:
pca_2 = PCA(n_components=2).fit(Xs)
Z = pca_2.transform(Xs)

fig, ax = plt.subplots(figsize=(5.2, 3.6))
for sp in df["species"].unique():
    mask = (df["species"] == sp).to_numpy()
    ax.scatter(Z[mask, 0], Z[mask, 1], s=10, alpha=0.6, label=sp)
ax.set_xlabel("PC1 (68.8% of variance)")
ax.set_ylabel("PC2 (19.3%)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
back = scaler.inverse_transform(pca_2.inverse_transform(Z))

print("bird 0, original:      ",
      [round(v, 1) for v in df[feats].iloc[0]])
print("bird 0, from 2 numbers:", [round(v, 1) for v in back[0]])

In [ ]:
import pandas as pd

print(pd.DataFrame(np.round(pca_2.components_, 2),
                   index=["PC1", "PC2"], columns=feats))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = df["species"]
for n in (1, 2, 3, 4):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("pca", PCA(n_components=n)),
                     ("model", LogisticRegression(max_iter=1000))])
    acc = cross_val_score(pipe, df[feats], y, cv=5).mean()
    print(f"{n} component(s): CV accuracy {acc:.3f}")

In [ ]:
pca = PCA(n_components=2)
Z = pca.fit_transform(Xs)

run_tests([
    ("each bird now has 2 coordinates", Z.shape, (342, 2)),
    ("variance the two axes carry",
     [round(float(r), 3) for r in pca.explained_variance_ratio_],
     [0.688, 0.193]),
])

In [ ]:
import math

def project(point, direction):
    return sum(p * d for p, d in zip(point, direction))

def variance_along(points, direction):
    coords = [project(p, direction) for p in points]
    mean = sum(coords) / len(coords)
    return sum((c - mean) ** 2 for c in coords) / len(coords)

cigar = [(-2.0, -1.0), (0.0, 0.0), (2.0, 1.0)]
along = (2 / math.sqrt(5), 1 / math.sqrt(5))
across = (-1 / math.sqrt(5), 2 / math.sqrt(5))

run_tests([
    ("projection onto the x-axis", project((3.0, 4.0), (1.0, 0.0)), 3.0),
    ("projection onto a tilted axis",
     round(project((10.0, 0.0), (0.6, 0.8)), 4), 6.0),
    ("spread along the cigar",
     round(variance_along(cigar, along), 4), 3.3333),
    ("no spread across it",
     round(variance_along(cigar, across), 4), 0.0),
], tol=1e-9)